Parte 1 - Priorização e Comunicação

Você está coordenando o time de sustentação e aparecem simultaneamente os seguintes chamados:

![quadro-chamados.png](./image_1784025853280.png "quadro-chamados.png")


Responda:
Em qual ordem você trataria os chamados e por quê?
O que comunicaria às áreas envolvidas em cada caso?


Ordem de atendimento

1. P1 - Falha na atualização do dashboard de Vendas

Primeiro, porque afeta a Diretoria e impede o acesso a informações críticas para decisão. É uma indisponibilidade completa.

2. P2 - Timeout no dataset de Operações

Segundo, porque impacta a operação e pode comprometer rotinas do dia a dia. Mesmo sendo intermitente, deve ser monitorado desde o início, em paralelo ao P1.

3. P3 - Ajuste de regra de cálculo no Power BI

Terceiro, porque pode gerar números incorretos para Finanças, mas parece ser uma correção funcional, não uma indisponibilidade total. Antes da alteração, validaria a regra com a área.

4. P4 - Criação de novo workspace

Por último, porque é uma solicitação de infraestrutura/evolução, sem impacto imediato em um processo existente.

Comunicação às áreas

P1 - Diretoria

Informaria que o chamado recebeu prioridade máxima, que a causa está sendo investigada e enviaria atualizações periódicas. Caso seja possível, disponibilizaria uma extração alternativa com os dados mais recentes.

P2 - Operações

Comunicaria que o problema está sendo analisado, pediria exemplos de horários e mensagens de erro e informaria possíveis alternativas temporárias, como reduzir o período consultado ou executar a atualização fora do horário de pico.

P3 - Finanças

Confirmaria o entendimento da regra, apresentaria o impacto esperado e combinaria uma validação antes da publicação. Também informaria se os relatórios atuais devem ser desconsiderados até a correção.

P4 - Marketing

Confirmaria o recebimento e explicaria que a solicitação será atendida após os incidentes de maior impacto. Solicitaria antecipadamente nome, responsáveis, permissões e finalidade do workspace para agilizar a execução.

**OBS.:** Um ponto importante é que eu avaliaria o impacto real: operações costuma ser onde o processo principal da empresa acontece. Se o timeout do P2 estiver impedindo atividades operacionais, atrasando entregas ou afetando clientes, ele pode virar prioridade 1, até acima do dashboard.

Se existe alternativa manual e o problema acontece poucas vezes, ele pode permanecer em segundo.

O dashboard da Diretoria é importante, mas talvez seja apenas informativo, enquanto Operações pode estar diretamente ligada à execução do negócio.

A depender de algumas validações, a ordem pode se tornar:

P2 - Operações, caso esteja bloqueando o trabalho.

P1 - Dashboard de Vendas, caso o dashboard da Diretoria esteja sem informação crítica.

P3 - Regra de cálculo, por risco de informação financeira incorreta.

P4 - Novo workspace, por ser uma melhoria sem incidente ativo.

Parte 2 - Power BI / BI

Cite duas práticas para melhorar a performance de um relatório no Power BI.

O refresh de um dataset está falhando por lentidão. Quais duas verificações básicas você faria primeiro?

1. Reduzir o volume de dados carregado

Remover colunas e linhas desnecessárias, filtrar períodos antigos e evitar tabelas muito detalhadas quando não forem necessárias.

2. Otimizar o modelo e as medidas

Utilizar modelo estrela, evitar relacionamentos muitos-para-muitos e bidirecionais sem necessidade e simplificar medidas DAX muito complexas.

3. Verificações básicas:

Verificar o histórico e a mensagem de erro do refresh, identificar em qual tabela ou etapa ocorreu a lentidão, o tempo de execução e se a falha foi realmente um timeout.

Verificar a fonte de dados, entender se está disponível e respondendo normalmente, além de avaliar se houve aumento no volume de dados carregado.

3.1 - Total de clientes cadastrados

Racional: calcula o total de clientes únicos cadastrados, desconsiderando onde o ID está null.

In [0]:
SELECT
    COUNT(DISTINCT customer_id) AS total_clientes_cadastrados
FROM customer_analytics.gold.customers
WHERE customer_id IS NOT NULL;

3.2 - Clientes por estado

Racional: calcula clientes únicos cadastrados, onde ID não está null e agrupa por estado.


In [0]:
SELECT
    state AS estado,
    COUNT(DISTINCT customer_id) AS total_clientes
FROM customer_analytics.gold.customers
WHERE customer_id IS NOT NULL
GROUP BY state;

3.3 - Clientes com transação por dia e por mês
*a)* Clientes únicos com transação por dia

Racional: calcula clientes únicos que realizaram transação em cada dia, ignorando IDs nulos e ordenando as datas.

In [0]:
SELECT
    CAST(transaction_date AS DATE) AS data_transacao,
    COUNT(DISTINCT customer_id) AS clientes_ativos
FROM customer_analytics.gold.orders
WHERE customer_id IS NOT NULL
GROUP BY CAST(transaction_date AS DATE)
ORDER BY data_transacao;

*b)* Clientes únicos com transação por mês

Racional: calcula clientes únicos que realizaram transação em cada mês, ignorando IDs nulos e ordenando os meses.

In [0]:
SELECT
    DATE_TRUNC('MONTH', CAST(transaction_date AS DATE)) AS mes,
    COUNT(DISTINCT customer_id) AS clientes_ativos
FROM customer_analytics.gold.orders
WHERE customer_id IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', CAST(transaction_date AS DATE))
ORDER BY mes;

3.4 - Participação de “km aplicativo Transporte”

Racional: calcula o total de transações, quantas pertencem à categoria KM Aplicativo Transporte e qual percentual elas representam no total.

In [0]:
SELECT
    COUNT(*) AS total_transacoes,
    COUNT_IF(LOWER(TRIM(transaction_category)) = 'km aplicativo transporte')
        AS transacoes_km_aplicativo_transporte,
    ROUND(
        100.0 * COUNT_IF(
            LOWER(TRIM(transaction_category)) = 'km aplicativo transporte'
        ) / COUNT(*),
        2
    ) AS percentual_transacoes
FROM customer_analytics.gold.orders;